In [3]:
import os

for filename in uploaded.keys():
    print(filename, "exists:", os.path.exists("/content/" + filename))

In [5]:
source_files = [
    "/content/IND.xlsx",
    "/content/USA.xlsx",
    "/content/AUS.xlsx"
]

In [6]:
import os

for filename in ["IND.xlsx", "USA.xlsx", "AUS.xlsx"]:
    print(filename, os.path.exists("/content/" + filename))

IND.xlsx True
USA.xlsx True
AUS.xlsx True


ImportError: cannot import name '_to_seq' from 'pyspark.sql.column' (/usr/local/lib/python3.13/dist-packages/pyspark/sql/column.py)

In [13]:
!pip install -q pyspark==3.5.3 delta-spark==3.2.1

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("SkyPoints-Assessment")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)
print("Spark session created successfully!")

Spark Version: 3.5.3
Spark session created successfully!


In [2]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [3]:
from google.colab import files

uploaded = files.upload()

Saving AUS.xlsx to AUS (1).xlsx
Saving IND.xlsx to IND (1).xlsx
Saving USA.xlsx to USA (1).xlsx


In [4]:
from pathlib import Path

base_path = Path("/content/skypoints")

folders = [
    "data/landing",
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/quarantine",
    "data/redemptions",
    "output"
]

for folder in folders:
    (base_path / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created!")

Project folders created!


In [5]:
import shutil

landing_path = base_path / "data" / "landing"

for filename in uploaded.keys():
    shutil.move(
        f"/content/{filename}",
        str(landing_path / filename)
    )

print("Files moved successfully!")

for file in landing_path.iterdir():
    print(file.name)

Files moved successfully!
USA (1).xlsx
IND (1).xlsx
AUS (1).xlsx


In [8]:
import os

for root, dirs, files in os.walk("/content"):
    for filename in files:
        if filename.endswith((".xlsx", ".xls")):
            print(os.path.join(root, filename))

/content/USA.xlsx
/content/IND.xlsx
/content/AUS.xlsx
/content/skypoints/data/landing/USA (1).xlsx
/content/skypoints/data/landing/IND (1).xlsx
/content/skypoints/data/landing/AUS (1).xlsx


In [10]:
import os

for root, dirs, files in os.walk("/content"):
    for filename in files:
        if filename in ["IND.xlsx", "USA.xlsx", "AUS.xlsx"]:
            print(os.path.join(root, filename))

/content/USA.xlsx
/content/IND.xlsx
/content/AUS.xlsx


In [11]:
from pathlib import Path
import shutil

base_path = Path("/content/skypoints")
landing_path = base_path / "data" / "landing"

landing_path.mkdir(parents=True, exist_ok=True)

for filename in ["IND.xlsx", "USA.xlsx", "AUS.xlsx"]:

    source = Path("/content") / filename

    if source.exists():
        destination = landing_path / filename
        shutil.move(str(source), str(destination))
        print("Moved:", filename)

    elif (landing_path / filename).exists():
        print("Already available:", filename)

    else:
        print("Missing:", filename, "- Please upload again")

Moved: IND.xlsx
Moved: USA.xlsx
Moved: AUS.xlsx


In [12]:
for file in landing_path.iterdir():
    print(file.name, file.exists())

USA (1).xlsx True
IND (1).xlsx True
USA.xlsx True
AUS (1).xlsx True
IND.xlsx True
AUS.xlsx True


In [13]:
from pathlib import Path

duplicate_files = [
    "IND (1).xlsx",
    "USA (1).xlsx",
    "AUS (1).xlsx"
]

for filename in duplicate_files:
    file_path = landing_path / filename

    if file_path.exists():
        file_path.unlink()
        print("Removed duplicate file:", filename)

print("\nRemaining files:")

for file in landing_path.iterdir():
    print(file.name)

Removed duplicate file: IND (1).xlsx
Removed duplicate file: USA (1).xlsx
Removed duplicate file: AUS (1).xlsx

Remaining files:
USA.xlsx
IND.xlsx
AUS.xlsx


In [14]:
import pandas as pd

for filename in ["IND.xlsx", "USA.xlsx", "AUS.xlsx"]:

    file_path = landing_path / filename

    print("\n" + "=" * 60)
    print("FILE:", filename)

    excel = pd.ExcelFile(file_path)

    for sheet in excel.sheet_names:

        df = pd.read_excel(file_path, sheet_name=sheet, dtype=str)

        print("\nSheet:", sheet)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        display(df.head(5))


FILE: IND.xlsx

Sheet: in
Shape: (3, 7)
Columns: ['ID', 'Name', 'DOB', 'TierCode', 'EnrollmentDate', 'Individual or Corporate', 'Flight Date']


,ID,Name,DOB,TierCode,EnrollmentDate,Individual or Corporate,Flight Date
0,1,Vikas,1998-12-01 00:00:00,SLV,2022-01-01 00:00:00,I,2022-06-15 00:00:00
1,2,Rahul,1982-08-13 00:00:00,GLD,2022-03-05 00:00:00,C,2022-03-10 00:00:00
2,3,Sameer,1952-08-13 00:00:00,GLD,2022-02-20 00:00:00,I,2022-02-25 00:00:00



FILE: USA.xlsx

Sheet: in
Shape: (3, 5)
Columns: ['ID', 'Name', 'TierCode', 'EnrollmentDate', 'FlightDate']


,ID,Name,TierCode,EnrollmentDate,FlightDate
0,1,Sam,PLT,6152022,8202022
1,2,John,SLV,1052022,1152022
2,3,Mike,GLD,12282021,12302021



FILE: AUS.xlsx

Sheet: Sheet1
Shape: (3, 6)
Columns: ['Unique ID', 'Member Name', 'Tier Type', 'Date of Birth', 'Date of Enrollment', 'Date of Flight']


,Unique ID,Member Name,Tier Type,Date of Birth,Date of Enrollment,Date of Flight
0,1,Mike,PLT,NaN,2022-05-11 00:00:00,2022-08-01 00:00:00
1,2,Jonnathan,GLD,1997-12-13 00:00:00,2021-13-13,2022-01-05 00:00:00
2,3,Cristina,SLV,1998-03-12 00:00:00,2022-03-12 00:00:00,2022-03-20 00:00:00


In [15]:
import pandas as pd
import json
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Source files
source_files = {
    "IND.xlsx": {
        "ID": "member_id",
        "Name": "member_name",
        "DOB": "dob_raw",
        "TierCode": "tier_code",
        "EnrollmentDate": "enrollment_date_raw",
        "Individual or Corporate": "member_type",
        "Flight Date": "flight_date_raw"
    },

    "USA.xlsx": {
        "ID": "member_id",
        "Name": "member_name",
        "TierCode": "tier_code",
        "EnrollmentDate": "enrollment_date_raw",
        "FlightDate": "flight_date_raw"
    },

    "AUS.xlsx": {
        "Unique ID": "member_id",
        "Member Name": "member_name",
        "Tier Type": "tier_code",
        "Date of Birth": "dob_raw",
        "Date of Enrollment": "enrollment_date_raw",
        "Date of Flight": "flight_date_raw"
    }
}

# Standard Bronze schema
columns = [
    "member_id",
    "member_name",
    "dob_raw",
    "tier_code",
    "enrollment_date_raw",
    "flight_date_raw",
    "member_type",
    "raw_record_json",
    "source_country",
    "source_file",
    "source_sheet"
]

schema = StructType([
    StructField(col, StringType(), True)
    for col in columns
])

bronze_dfs = []

for filename, mapping in source_files.items():

    file_path = landing_path / filename

    excel = pd.ExcelFile(file_path)

    for sheet_name in excel.sheet_names:

        pdf = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            dtype=object
        )

        records = []

        for _, row in pdf.iterrows():

            record = {col: None for col in columns}

            # Preserve original row values
            raw_record = {}

            for original_col, value in row.items():

                if pd.isna(value):
                    raw_record[original_col] = None

                elif hasattr(value, "isoformat"):
                    raw_record[original_col] = value.isoformat()

                else:
                    raw_record[original_col] = str(value)

            # Map source columns to standard columns
            for original_col, standard_col in mapping.items():

                value = row[original_col]

                if pd.isna(value):
                    record[standard_col] = None

                elif hasattr(value, "isoformat"):
                    record[standard_col] = value.isoformat()

                else:
                    record[standard_col] = str(value)

            record["raw_record_json"] = json.dumps(raw_record)
            record["source_country"] = Path(filename).stem
            record["source_file"] = filename
            record["source_sheet"] = sheet_name

            records.append(record)

        df = spark.createDataFrame(records, schema=schema)

        df = df.withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )

        bronze_dfs.append(df)

# Combine all country files
bronze_df = bronze_dfs[0]

for df in bronze_dfs[1:]:
    bronze_df = bronze_df.unionByName(
        df,
        allowMissingColumns=True
    )

print("Bronze ingestion completed!")
print("Total records:", bronze_df.count())

bronze_df.show(truncate=False)

Bronze ingestion completed!
Total records: 9
+---------+-----------+-------------------+---------+-------------------+-------------------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-----------+------------+--------------------------+
|member_id|member_name|dob_raw            |tier_code|enrollment_date_raw|flight_date_raw    |member_type|raw_record_json                                                                                                                                                                                |source_country|source_file|source_sheet|ingestion_timestamp       |
+---------+-----------+-------------------+---------+-------------------+-------------------+-----------+------------------------------------------------------------------------------------------------------------------------

In [16]:
bronze_path = "/content/skypoints/data/bronze/member_raw"

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(bronze_path)

print("Bronze Delta table saved successfully!")

Bronze Delta table saved successfully!


In [17]:
bronze_read = spark.read.format("delta").load(bronze_path)

print("Total Bronze records:", bronze_read.count())

bronze_read.printSchema()

bronze_read.groupBy("source_country").count().show()

bronze_read.show(10, truncate=False)

Total Bronze records: 9
root
 |-- member_id: string (nullable = true)
 |-- member_name: string (nullable = true)
 |-- dob_raw: string (nullable = true)
 |-- tier_code: string (nullable = true)
 |-- enrollment_date_raw: string (nullable = true)
 |-- flight_date_raw: string (nullable = true)
 |-- member_type: string (nullable = true)
 |-- raw_record_json: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

+--------------+-----+
|source_country|count|
+--------------+-----+
|           AUS|    3|
|           IND|    3|
|           USA|    3|
+--------------+-----+

+---------+-----------+-------------------+---------+-------------------+-------------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
bronze_df = spark.read.format("delta").load(
    "/content/skypoints/data/bronze/member_raw"
)

print("Bronze records:", bronze_df.count())

Bronze records: 9


In [19]:
from pyspark.sql import functions as F

# Disable ANSI date parsing exceptions so invalid dates become NULL
spark.conf.set("spark.sql.ansi.enabled", "false")

df = bronze_df

# Clean raw date strings
date_columns = ["dob_raw", "enrollment_date_raw", "flight_date_raw"]

for col_name in date_columns:
    df = df.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

# Prepare USA compact dates
for col_name in date_columns:

    normalized = F.regexp_replace(F.col(col_name), r"\.0$", "")

    normalized = F.when(
        (F.col("source_country") == "USA") &
        (F.length(normalized) == 7),
        F.concat(F.lit("0"), normalized)
    ).otherwise(normalized)

    df = df.withColumn(
        col_name + "_normalized",
        normalized
    )

# Parse dates
def parse_date(column_name):

    raw = F.col(column_name)
    normalized = F.col(column_name + "_normalized")

    return F.when(
        F.col("source_country") == "USA",
        F.to_date(normalized, "MMddyyyy")
    ).otherwise(
        F.to_date(F.substring(raw, 1, 10), "yyyy-MM-dd")
    )

df = (
    df.withColumn("dob", parse_date("dob_raw"))
      .withColumn("enrollment_date", parse_date("enrollment_date_raw"))
      .withColumn("flight_date", parse_date("flight_date_raw"))
)

df.select(
    "source_country",
    "member_id",
    "member_name",
    "dob_raw",
    "dob",
    "enrollment_date_raw",
    "enrollment_date",
    "flight_date_raw",
    "flight_date"
).show(20, truncate=False)

+--------------+---------+-----------+-------------------+----------+-------------------+---------------+-------------------+-----------+
|source_country|member_id|member_name|dob_raw            |dob       |enrollment_date_raw|enrollment_date|flight_date_raw    |flight_date|
+--------------+---------+-----------+-------------------+----------+-------------------+---------------+-------------------+-----------+
|IND           |1        |Vikas      |1998-12-01T00:00:00|1998-12-01|2022-01-01T00:00:00|2022-01-01     |2022-06-15T00:00:00|2022-06-15 |
|AUS           |1        |Mike       |NULL               |NULL      |2022-05-11T00:00:00|2022-05-11     |2022-08-01T00:00:00|2022-08-01 |
|IND           |2        |Rahul      |1982-08-13T00:00:00|1982-08-13|2022-03-05T00:00:00|2022-03-05     |2022-03-10T00:00:00|2022-03-10 |
|IND           |3        |Sameer     |1952-08-13T00:00:00|1952-08-13|2022-02-20T00:00:00|2022-02-20     |2022-02-25T00:00:00|2022-02-25 |
|AUS           |2        |Jonnatha

In [20]:
processing_date = F.to_date(F.lit("2026-11-23"))

df = df.withColumn(
    "age",
    F.when(
        F.col("dob").isNotNull() &
        (F.col("dob") <= processing_date),
        F.floor(
            F.months_between(processing_date, F.col("dob")) / 12
        ).cast("int")
    )
)

df.select(
    "member_id",
    "member_name",
    "dob",
    "age"
).show()

+---------+-----------+----------+----+
|member_id|member_name|       dob| age|
+---------+-----------+----------+----+
|        1|      Vikas|1998-12-01|  27|
|        1|       Mike|      NULL|NULL|
|        2|      Rahul|1982-08-13|  44|
|        3|     Sameer|1952-08-13|  74|
|        2|  Jonnathan|1997-12-13|  28|
|        3|   Cristina|1998-03-12|  28|
|        1|        Sam|      NULL|NULL|
|        2|       John|      NULL|NULL|
|        3|       Mike|      NULL|NULL|
+---------+-----------+----------+----+



In [21]:
df = df.withColumn(
    "days_since_last_flight",
    F.datediff(processing_date, F.col("flight_date"))
)

df = df.withColumn(
    "stale_member",
    F.when(
        F.col("flight_date").isNull(),
        F.lit(None).cast("boolean")
    ).otherwise(
        F.col("days_since_last_flight") > 90
    )
)

df.select(
    "member_id",
    "member_name",
    "flight_date",
    "days_since_last_flight",
    "stale_member"
).show(truncate=False)

+---------+-----------+-----------+----------------------+------------+
|member_id|member_name|flight_date|days_since_last_flight|stale_member|
+---------+-----------+-----------+----------------------+------------+
|1        |Vikas      |2022-06-15 |1622                  |true        |
|1        |Mike       |2022-08-01 |1575                  |true        |
|2        |Rahul      |2022-03-10 |1719                  |true        |
|3        |Sameer     |2022-02-25 |1732                  |true        |
|2        |Jonnathan  |2022-01-05 |1783                  |true        |
|3        |Cristina   |2022-03-20 |1709                  |true        |
|1        |Sam        |2022-08-20 |1556                  |true        |
|2        |John       |2022-01-15 |1773                  |true        |
|3        |Mike       |2021-12-30 |1789                  |true        |
+---------+-----------+-----------+----------------------+------------+



In [22]:
df = df.withColumn(
    "validation_errors",
    F.concat_ws(
        " | ",

        F.when(
            F.col("member_id").isNull() |
            (F.trim(F.col("member_id")) == ""),
            "Missing Member ID"
        ),

        F.when(
            F.col("member_name").isNull() |
            (F.trim(F.col("member_name")) == ""),
            "Missing Member Name"
        ),

        F.when(
            F.col("dob").isNull(),
            "Missing or Invalid DOB"
        ),

        F.when(
            F.col("dob") > processing_date,
            "Future DOB"
        ),

        F.when(
            F.col("tier_code").isNull() |
            (F.trim(F.col("tier_code")) == ""),
            "Missing Tier Code"
        ),

        F.when(
            F.col("enrollment_date").isNull(),
            "Missing or Invalid Enrollment Date"
        ),

        F.when(
            F.col("flight_date").isNull(),
            "Missing or Invalid Flight Date"
        ),

        F.when(
            F.col("enrollment_date") > processing_date,
            "Future Enrollment Date"
        ),

        F.when(
            F.col("flight_date") > processing_date,
            "Future Flight Date"
        )
    )
)

df = df.withColumn(
    "is_valid",
    F.col("validation_errors") == ""
)

df.groupBy("is_valid").count().show()

df.select(
    "source_country",
    "member_id",
    "member_name",
    "validation_errors",
    "is_valid"
).show(truncate=False)

+--------+-----+
|is_valid|count|
+--------+-----+
|    true|    4|
|   false|    5|
+--------+-----+

+--------------+---------+-----------+----------------------------------+--------+
|source_country|member_id|member_name|validation_errors                 |is_valid|
+--------------+---------+-----------+----------------------------------+--------+
|IND           |1        |Vikas      |                                  |true    |
|AUS           |1        |Mike       |Missing or Invalid DOB            |false   |
|IND           |2        |Rahul      |                                  |true    |
|IND           |3        |Sameer     |                                  |true    |
|AUS           |2        |Jonnathan  |Missing or Invalid Enrollment Date|false   |
|AUS           |3        |Cristina   |                                  |true    |
|USA           |1        |Sam        |Missing or Invalid DOB            |false   |
|USA           |2        |John       |Missing or Invalid DOB       

In [23]:
silver_df = df.filter(F.col("is_valid") == True)

quarantine_df = df.filter(F.col("is_valid") == False)

print("Valid records:", silver_df.count())
print("Invalid records:", quarantine_df.count())

Valid records: 4
Invalid records: 5


In [24]:
quarantine_df.select(
    "source_country",
    "member_id",
    "member_name",
    "dob_raw",
    "enrollment_date_raw",
    "validation_errors"
).show(truncate=False)

+--------------+---------+-----------+-------------------+-------------------+----------------------------------+
|source_country|member_id|member_name|dob_raw            |enrollment_date_raw|validation_errors                 |
+--------------+---------+-----------+-------------------+-------------------+----------------------------------+
|AUS           |1        |Mike       |NULL               |2022-05-11T00:00:00|Missing or Invalid DOB            |
|AUS           |2        |Jonnathan  |1997-12-13T00:00:00|2021-13-13         |Missing or Invalid Enrollment Date|
|USA           |1        |Sam        |NULL               |6152022            |Missing or Invalid DOB            |
|USA           |2        |John       |NULL               |1052022            |Missing or Invalid DOB            |
|USA           |3        |Mike       |NULL               |12282021           |Missing or Invalid DOB            |
+--------------+---------+-----------+-------------------+-------------------+----------

In [25]:
silver_path = "/content/skypoints/data/silver/members"
quarantine_path = "/content/skypoints/data/quarantine/invalid_members"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(quarantine_path)

print("Silver and Quarantine tables saved successfully!")

Silver and Quarantine tables saved successfully!


In [26]:
silver_read = spark.read.format("delta").load(silver_path)

print("Silver record count:", silver_read.count())

silver_read.printSchema()

silver_read.show(truncate=False)

Silver record count: 4
root
 |-- member_id: string (nullable = true)
 |-- member_name: string (nullable = true)
 |-- dob_raw: string (nullable = true)
 |-- tier_code: string (nullable = true)
 |-- enrollment_date_raw: string (nullable = true)
 |-- flight_date_raw: string (nullable = true)
 |-- member_type: string (nullable = true)
 |-- raw_record_json: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- dob_raw_normalized: string (nullable = true)
 |-- enrollment_date_raw_normalized: string (nullable = true)
 |-- flight_date_raw_normalized: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- enrollment_date: date (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- days_since_last_flight: integer (nullable = true)
 |-- stale_member: boolean (nullable = true)
 |-- val

In [27]:
from pyspark.sql import functions as F

# Recalculate validation errors
df = df.withColumn(
    "validation_errors",
    F.concat_ws(
        " | ",

        F.when(
            F.col("member_id").isNull() |
            (F.trim(F.col("member_id")) == ""),
            "Missing Member ID"
        ),

        F.when(
            F.col("member_name").isNull() |
            (F.trim(F.col("member_name")) == ""),
            "Missing Member Name"
        ),

        # DOB is optional, but invalid DOB is rejected
        F.when(
            F.col("dob_raw").isNotNull() &
            (F.trim(F.col("dob_raw")) != "") &
            F.col("dob").isNull(),
            "Invalid DOB"
        ),

        F.when(
            F.col("dob") > processing_date,
            "Future DOB"
        ),

        F.when(
            F.col("tier_code").isNull() |
            (F.trim(F.col("tier_code")) == ""),
            "Missing Tier Code"
        ),

        F.when(
            F.col("enrollment_date").isNull(),
            "Missing or Invalid Enrollment Date"
        ),

        F.when(
            F.col("flight_date").isNull(),
            "Missing or Invalid Flight Date"
        ),

        F.when(
            F.col("enrollment_date") > processing_date,
            "Future Enrollment Date"
        ),

        F.when(
            F.col("flight_date") > processing_date,
            "Future Flight Date"
        )
    )
)

df = df.withColumn(
    "is_valid",
    F.col("validation_errors") == ""
)

In [28]:
silver_df = df.filter(F.col("is_valid") == True)

quarantine_df = df.filter(F.col("is_valid") == False)

print("Total records:", df.count())
print("Valid records:", silver_df.count())
print("Invalid records:", quarantine_df.count())

quarantine_df.select(
    "source_country",
    "member_id",
    "member_name",
    "dob_raw",
    "enrollment_date_raw",
    "validation_errors"
).show(truncate=False)

Total records: 9
Valid records: 8
Invalid records: 1
+--------------+---------+-----------+-------------------+-------------------+----------------------------------+
|source_country|member_id|member_name|dob_raw            |enrollment_date_raw|validation_errors                 |
+--------------+---------+-----------+-------------------+-------------------+----------------------------------+
|AUS           |2        |Jonnathan  |1997-12-13T00:00:00|2021-13-13         |Missing or Invalid Enrollment Date|
+--------------+---------+-----------+-------------------+-------------------+----------------------------------+



In [29]:
silver_path = "/content/skypoints/data/silver/members"
quarantine_path = "/content/skypoints/data/quarantine/invalid_members"

silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

quarantine_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(quarantine_path)

print("Silver and Quarantine updated successfully!")

Silver and Quarantine updated successfully!


In [30]:
from pyspark.sql import functions as F

# Read validated Silver data
silver_df = spark.read.format("delta").load(silver_path)

# Select final Gold columns
gold_base = silver_df.select(
    "member_id",
    "member_name",
    "dob",
    "age",
    "tier_code",
    "enrollment_date",
    "flight_date",
    "member_type",
    "days_since_last_flight",
    "stale_member",
    "source_country"
)

# Create country-specific Gold tables
country_tables = [
    ("IND", "gold_ind_members"),
    ("USA", "gold_usa_members"),
    ("AUS", "gold_aus_members")
]

for country, table_name in country_tables:

    country_df = gold_base.filter(
        F.col("source_country") == country
    )

    gold_path = f"/content/skypoints/data/gold/{table_name}"

    country_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(gold_path)

    print(f"{table_name} created successfully")
    print("Records:", country_df.count())

gold_ind_members created successfully
Records: 3
gold_usa_members created successfully
Records: 3
gold_aus_members created successfully
Records: 2


In [31]:
for country, table_name in country_tables:

    gold_path = f"/content/skypoints/data/gold/{table_name}"

    print("\n==========", table_name, "==========")

    df_gold = spark.read.format("delta").load(gold_path)

    df_gold.show(truncate=False)
    print("Total records:", df_gold.count())


========== gold_ind_members ==========
+---------+-----------+----------+---+---------+---------------+-----------+-----------+----------------------+------------+--------------+
|member_id|member_name|dob       |age|tier_code|enrollment_date|flight_date|member_type|days_since_last_flight|stale_member|source_country|
+---------+-----------+----------+---+---------+---------------+-----------+-----------+----------------------+------------+--------------+
|1        |Vikas      |1998-12-01|27 |SLV      |2022-01-01     |2022-06-15 |I          |1622                  |true        |IND           |
|2        |Rahul      |1982-08-13|44 |GLD      |2022-03-05     |2022-03-10 |C          |1719                  |true        |IND           |
|3        |Sameer     |1952-08-13|74 |GLD      |2022-02-20     |2022-02-25 |I          |1732                  |true        |IND           |
+---------+-----------+----------+---+---------+---------------+-----------+-----------+----------------------+---------

In [32]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

member_schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Mem_Id", StringType(), True),
    StructField("Enroll_Dt", StringType(), True),
    StructField("Flight_Dt", StringType(), True),
    StructField("TIER", StringType(), True),
    StructField("Agent_Name", StringType(), True),
    StructField("State", StringType(), True),
    StructField("County", StringType(), True),
    StructField("DOB", StringType(), True),
    StructField("FLAG", StringType(), True)
])

member_data = [
    ("Elena", "223457", "20101012", "20121013", "GLD", "Sam", "CA", "USA", "3051985", "A"),
    ("Ravi", "223458", "20101012", "20121013", "SLV", "Sam", "TN", "IND", "3051985", "A"),
    ("Mateo", "223459", "20101012", "20121013", "GLD", "Sam", "NCR", "PHIL", "3051985", "A"),
    ("Nora", "22345", "20101012", "20121013", "PLT", "Sam", "ONT", "CAN", "3051985", "A"),
    ("Jacob", "2256", "20101012", "20121013", "SLV", "Sam", "VIC", "AU", "3051985", "A")
]

member_staging_df = spark.createDataFrame(member_data, member_schema)

member_staging_df.show(truncate=False)

+-----+------+---------+---------+----+----------+-----+------+-------+----+
|Name |Mem_Id|Enroll_Dt|Flight_Dt|TIER|Agent_Name|State|County|DOB    |FLAG|
+-----+------+---------+---------+----+----------+-----+------+-------+----+
|Elena|223457|20101012 |20121013 |GLD |Sam       |CA   |USA   |3051985|A   |
|Ravi |223458|20101012 |20121013 |SLV |Sam       |TN   |IND   |3051985|A   |
|Mateo|223459|20101012 |20121013 |GLD |Sam       |NCR  |PHIL  |3051985|A   |
|Nora |22345 |20101012 |20121013 |PLT |Sam       |ONT  |CAN   |3051985|A   |
|Jacob|2256  |20101012 |20121013 |SLV |Sam       |VIC  |AU    |3051985|A   |
+-----+------+---------+---------+----+----------+-----+------+-------+----+



In [33]:
import json

redemption_data = {
    "member_id": "223457",
    "feed_date": "20240115",
    "redemptions": [
        {
            "txn_id": "RX10091",
            "txn_date": "20240110",
            "partner": "AeroLink",
            "miles_redeemed": 12000,
            "status": "COMPLETED"
        },
        {
            "txn_id": "RX10092",
            "txn_date": "20240113",
            "partner": "SkyPoints",
            "miles_redeemed": 5000,
            "status": "PENDING"
        }
    ]
}

json_path = "/content/redemption_feed.json"

with open(json_path, "w") as file:
    json.dump(redemption_data, file, indent=4)

print("Redemption JSON file created successfully!")

Redemption JSON file created successfully!


In [34]:
from pyspark.sql import functions as F

# Read JSON file
redemption_df = (
    spark.read
    .option("multiLine", True)
    .json(json_path)
)

print("Original JSON structure:")
redemption_df.printSchema()

# Flatten nested redemptions array
flattened_df = redemption_df.select(
    F.col("member_id"),
    F.col("feed_date"),
    F.explode("redemptions").alias("redemption")
)

# Extract individual transaction fields
redemption_flat_df = flattened_df.select(
    "member_id",
    "feed_date",
    F.col("redemption.txn_id").alias("txn_id"),
    F.col("redemption.txn_date").alias("txn_date"),
    F.col("redemption.partner").alias("partner"),
    F.col("redemption.miles_redeemed").alias("miles_redeemed"),
    F.col("redemption.status").alias("status")
)

redemption_flat_df.show(truncate=False)


Original JSON structure:
root
 |-- feed_date: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- redemptions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- miles_redeemed: long (nullable = true)
 |    |    |-- partner: string (nullable = true)
 |    |    |-- status: string (nullable = true)
 |    |    |-- txn_date: string (nullable = true)
 |    |    |-- txn_id: string (nullable = true)

+---------+---------+-------+--------+---------+--------------+---------+
|member_id|feed_date|txn_id |txn_date|partner  |miles_redeemed|status   |
+---------+---------+-------+--------+---------+--------------+---------+
|223457   |20240115 |RX10091|20240110|AeroLink |12000         |COMPLETED|
|223457   |20240115 |RX10092|20240113|SkyPoints|5000          |PENDING  |
+---------+---------+-------+--------+---------+--------------+---------+



In [35]:
joined_df = redemption_flat_df.join(
    member_staging_df,
    redemption_flat_df.member_id == member_staging_df.Mem_Id,
    "left"
)

final_redemption_df = joined_df.select(
    redemption_flat_df.member_id,
    "Name",
    "TIER",
    "County",
    "feed_date",
    "txn_id",
    "txn_date",
    "partner",
    "miles_redeemed",
    "status"
)

final_redemption_df.show(truncate=False)

+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+
|member_id|Name |TIER|County|feed_date|txn_id |txn_date|partner  |miles_redeemed|status   |
+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+
|223457   |Elena|GLD |USA   |20240115 |RX10091|20240110|AeroLink |12000         |COMPLETED|
|223457   |Elena|GLD |USA   |20240115 |RX10092|20240113|SkyPoints|5000          |PENDING  |
+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+



In [36]:
gold_redemption_path = "/content/skypoints/data/gold/gold_redemptions"

final_redemption_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_redemption_path)

print("Gold redemption table saved successfully!")

Gold redemption table saved successfully!


In [37]:
gold_redemption_df = spark.read.format("delta").load(
    gold_redemption_path
)

gold_redemption_df.show(truncate=False)

print("Total redemption records:", gold_redemption_df.count())

+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+
|member_id|Name |TIER|County|feed_date|txn_id |txn_date|partner  |miles_redeemed|status   |
+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+
|223457   |Elena|GLD |USA   |20240115 |RX10091|20240110|AeroLink |12000         |COMPLETED|
|223457   |Elena|GLD |USA   |20240115 |RX10092|20240113|SkyPoints|5000          |PENDING  |
+---------+-----+----+------+---------+-------+--------+---------+--------------+---------+

Total redemption records: 2


In [38]:
# Mandatory field validation
invalid_redemptions = final_redemption_df.filter(
    F.col("member_id").isNull() |
    F.col("txn_id").isNull() |
    F.col("txn_date").isNull() |
    F.col("miles_redeemed").isNull() |
    F.col("status").isNull()
)

# Duplicate transaction validation
duplicate_txns = (
    final_redemption_df
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
)

# Invalid miles validation
invalid_miles = final_redemption_df.filter(
    F.col("miles_redeemed") <= 0
)

print("Invalid mandatory records:", invalid_redemptions.count())
print("Duplicate transactions:", duplicate_txns.count())
print("Invalid miles records:", invalid_miles.count())

Invalid mandatory records: 0
Duplicate transactions: 0
Invalid miles records: 0


In [39]:
from pyspark.sql import functions as F

# 1. Member source count
source_count = member_staging_df.count()

# 2. Member target count
target_count = final_redemption_df.select(
    "member_id"
).distinct().count()

# 3. Redemption transaction count
redemption_count = final_redemption_df.count()

# 4. Check unmatched members
unmatched_members = final_redemption_df.filter(
    F.col("Name").isNull()
)

# 5. Check duplicate transaction IDs
duplicate_transactions = (
    final_redemption_df
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
)

# 6. Check invalid redemption amounts
invalid_amounts = final_redemption_df.filter(
    F.col("miles_redeemed") <= 0
)

print("===== SOURCE TO TARGET VALIDATION =====")
print("Member source count:", source_count)
print("Distinct matched members:", target_count)
print("Redemption transaction count:", redemption_count)
print("Unmatched redemption records:", unmatched_members.count())
print("Duplicate transactions:", duplicate_transactions.count())
print("Invalid redemption amounts:", invalid_amounts.count())

===== SOURCE TO TARGET VALIDATION =====
Member source count: 5
Distinct matched members: 1
Redemption transaction count: 2
Unmatched redemption records: 0
Duplicate transactions: 0
Invalid redemption amounts: 0


In [40]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

transfer_data = [
    ("223457", "Elena", "USA", "2024-01-10 10:00:00"),
    ("223457", "Elena", "IND", "2024-01-20 12:00:00"),
    ("223458", "Ravi",  "IND", "2024-01-15 09:00:00"),
    ("223459", "Mateo", "PHIL", "2024-01-18 11:00:00")
]

transfer_columns = [
    "member_id", "member_name", "country", "source_updated_at"
]

transfer_df = spark.createDataFrame(
    transfer_data, transfer_columns
)

window_spec = Window.partitionBy("member_id").orderBy(
    F.to_timestamp("source_updated_at").desc()
)

latest_member_df = (
    transfer_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_member_df.show(truncate=False)

+---------+-----------+-------+-------------------+
|member_id|member_name|country|source_updated_at  |
+---------+-----------+-------+-------------------+
|223457   |Elena      |IND    |2024-01-20 12:00:00|
|223458   |Ravi       |IND    |2024-01-15 09:00:00|
|223459   |Mateo      |PHIL   |2024-01-18 11:00:00|
+---------+-----------+-------+-------------------+



In [41]:
%%writefile /content/test_etl.py

import json
import pytest

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


@pytest.fixture(scope="session")
def spark():
    return (
        SparkSession.builder
        .master("local[*]")
        .appName("SkyPoints-PyTest")
        .getOrCreate()
    )


def flatten_redemptions(spark, json_path):

    df = (
        spark.read
        .option("multiLine", True)
        .json(json_path)
    )

    return df.select(
        "member_id",
        "feed_date",
        F.explode("redemptions").alias("r")
    ).select(
        "member_id",
        "feed_date",
        F.col("r.txn_id").alias("txn_id"),
        F.col("r.txn_date").alias("txn_date"),
        F.col("r.partner").alias("partner"),
        F.col("r.miles_redeemed").alias("miles_redeemed"),
        F.col("r.status").alias("status")
    )


def validate_redemptions(df):

    invalid = df.filter(
        F.col("member_id").isNull() |
        F.col("txn_id").isNull() |
        F.col("txn_date").isNull() |
        F.col("miles_redeemed").isNull() |
        F.col("status").isNull()
    ).count()

    duplicates = (
        df.groupBy("txn_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    invalid_miles = df.filter(
        F.col("miles_redeemed") <= 0
    ).count()

    return invalid, duplicates, invalid_miles


def get_latest_members(df):

    window_spec = Window.partitionBy("member_id").orderBy(
        F.to_timestamp("source_updated_at").desc()
    )

    return (
        df.withColumn(
            "row_num",
            F.row_number().over(window_spec)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )


def test_json_flattening(spark, tmp_path):

    data = {
        "member_id": "223457",
        "feed_date": "20240115",
        "redemptions": [
            {
                "txn_id": "RX10091",
                "txn_date": "20240110",
                "partner": "AeroLink",
                "miles_redeemed": 12000,
                "status": "COMPLETED"
            },
            {
                "txn_id": "RX10092",
                "txn_date": "20240113",
                "partner": "SkyPoints",
                "miles_redeemed": 5000,
                "status": "PENDING"
            }
        ]
    }

    json_path = tmp_path / "redemption.json"
    json_path.write_text(json.dumps(data))

    result = flatten_redemptions(spark, str(json_path))

    assert result.count() == 2

    assert set(
        row.txn_id for row in result.collect()
    ) == {"RX10091", "RX10092"}


def test_redemption_validation(spark):

    data = [
        ("223457", "RX10091", "20240110", 12000, "COMPLETED"),
        ("223457", "RX10092", "20240113", 5000, "PENDING")
    ]

    columns = [
        "member_id", "txn_id", "txn_date",
        "miles_redeemed", "status"
    ]

    df = spark.createDataFrame(data, columns)

    invalid, duplicates, invalid_miles = validate_redemptions(df)

    assert invalid == 0
    assert duplicates == 0
    assert invalid_miles == 0


def test_latest_record_wins(spark):

    data = [
        ("223457", "Elena", "USA", "2024-01-10 10:00:00"),
        ("223457", "Elena", "IND", "2024-01-20 12:00:00")
    ]

    columns = [
        "member_id", "member_name",
        "country", "source_updated_at"
    ]

    df = spark.createDataFrame(data, columns)

    result = get_latest_members(df)

    assert result.count() == 1

    assert result.first()["country"] == "IND"

Writing /content/test_etl.py


In [42]:
import pytest

result = pytest.main(["-q", "/content/test_etl.py"])

print("PyTest execution completed.")

...                                                                      [100%]
3 passed in 2.37s
PyTest execution completed.


In [43]:
print("========== SKYPOINTS PIPELINE SUMMARY ==========")

print("Bronze records:",
      spark.read.format("delta")
      .load("/content/skypoints/data/bronze/member_raw")
      .count())

print("Silver valid records:",
      spark.read.format("delta")
      .load(silver_path)
      .count())

print("Quarantine records:",
      spark.read.format("delta")
      .load(quarantine_path)
      .count())

print("India Gold records:",
      spark.read.format("delta")
      .load("/content/skypoints/data/gold/gold_ind_members")
      .count())

print("USA Gold records:",
      spark.read.format("delta")
      .load("/content/skypoints/data/gold/gold_usa_members")
      .count())

print("Australia Gold records:",
      spark.read.format("delta")
      .load("/content/skypoints/data/gold/gold_aus_members")
      .count())

print("Gold redemption records:",
      spark.read.format("delta")
      .load(gold_redemption_path)
      .count())

========== SKYPOINTS PIPELINE SUMMARY ==========
Bronze records: 9
Silver valid records: 8
Quarantine records: 1
India Gold records: 3
USA Gold records: 3
Australia Gold records: 2
Gold redemption records: 2
